# Final model metric standard — reusable evaluator

`BrainCDSS_최종_모델_평가_Metric_표준_v1.0`의 key를 그대로 사용한다.

- 입력: **같은 test split**에서 수행한 full-volume, original-space binary NIfTI prediction
- 출력: 모델별 `evaluation_metrics.json`
- CT/MRI small lesion: `< 5.0 mL`; MRA small lesion metric: `null`
- validation 결과를 실행할 수는 있지만 `warning`이 남으며 최종 운영 성능으로 사용하면 안 된다.

재사용 방법: 아래 `MODEL_CATALOG`에 새 모델의 export 경로를 한 번 등록하고, 이후 `SELECTED_MODEL_IDS`에서 모델 ID만 바꾼다.


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
from IPython.display import display

REPO = Path("/home/stu03/workspace/model")
PYTHON = REPO / ".venv/bin/python"
EVALUATOR = REPO / "scripts/evaluation/standard_final_metrics.py"
assert PYTHON.is_file() and EVALUATOR.is_file()

CT = "HEMORRHAGIC_STROKE_CT"
MRI = "ISCHEMIC_STROKE_DWI_ADC"
MRA = "CEREBRAL_ANEURYSM_MRA"


## Model catalog

`prediction_dir`에는 threshold를 확정한 binary mask만 둔다. 모든 case ID는 GT 파일명과 같아야 한다. test split case 수를 고정하려면 `case_list`를 지정한다.

새 모델은 아래 형식으로 한 줄만 추가하면 된다. 모델 ID 선택만으로 같은 표준 JSON을 다시 만들 수 있다.


In [ ]:
# Final operating candidates only. Each directory must contain full-volume test masks.
MODEL_CATALOG = {
    "H-T02-E300": {
        "analysis_type": CT,
        "prediction_dir": REPO / "runs/final_test_exports/H-T02-E300/predictions",
        "prediction_template": "{case_id}.nii.gz",
        "label_dir": REPO / "../datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_raw/Dataset001_ICHBinary/labelsTs",
        "case_list": None, "split_name": "test",
    },
    "I-E03": {
        "analysis_type": MRI,
        "prediction_dir": REPO / "runs/final_test_exports/I-E03/predictions",
        "prediction_template": "{case_id}.nii.gz",
        "label_dir": REPO / "../datasets/ischemic/nnunet/nnUNet_raw/Dataset502_ISLES2022/labelsTs",
        "case_list": None, "split_name": "test",
    },
    "A-B04": {
        "analysis_type": MRA,
        "prediction_dir": REPO / "runs/final_test_exports/A-B04/predictions",
        "prediction_template": "{case_id}.nii.gz",
        "label_dir": REPO / "../datasets/aneurysm/v2/model_ready/nnunet/nnUNet_raw/Dataset503_TopAneuMRABinary/test_gt",
        "case_list": None, "split_name": "test",
    },
}
SELECTED_MODEL_IDS = ["H-T02-E300", "I-E03", "A-B04"]
for model_id in SELECTED_MODEL_IDS:
    assert model_id in MODEL_CATALOG, f"Unknown model ID: {model_id}"
MODEL_CATALOG


## Execute standard evaluation

This does not run model inference. It evaluates exported full-volume masks. For an unregistered model, first export masks into `runs/final_test_exports/<MODEL_ID>/predictions/`, add one catalog entry, then select the ID.


In [ ]:
reports = []
for model_id in SELECTED_MODEL_IDS:
    spec = MODEL_CATALOG[model_id]
    output = REPO / "runs/final_evaluations" / model_id / "evaluation_metrics.json"
    command = [
        str(PYTHON), str(EVALUATOR),
        "--analysis-type", spec["analysis_type"],
        "--model-id", model_id,
        "--split-name", spec["split_name"],
        "--label-dir", str(spec["label_dir"]),
        "--prediction-dir", str(spec["prediction_dir"]),
        "--prediction-template", spec["prediction_template"],
        "--output", str(output),
    ]
    if spec.get("case_list"):
        command += ["--case-list", str(spec["case_list"])]
    print("\n", " ".join(command))
    subprocess.run(command, cwd=REPO, check=True)
    reports.append(json.loads(output.read_text()))
print(f"Created {len(reports)} evaluation_metrics.json file(s).")


In [ ]:
KEYS = [
    "model_id", "analysis_type", "evaluation_split", "warning",
    "test_macro_dice_positive", "test_macro_iou_positive",
    "test_voxel_recall_positive", "test_voxel_precision_positive",
    "test_case_detection_rate", "test_negative_case_fp_rate",
    "test_lesion_recall", "test_lesion_precision", "test_lesion_f1",
    "test_fp_per_case", "test_small_lesion_dice", "test_small_lesion_recall",
    "test_mean_assd_mm", "test_hd95_mm", "total_cases", "positive_cases",
    "negative_cases", "total_gt_lesions", "total_pred_lesions", "small_gt_lesions",
]
display(pd.DataFrame([{key: report.get(key) for key in KEYS} for report in reports]))

for report in reports:
    path = REPO / "runs/final_evaluations" / report["model_id"] / "evaluation_metrics.json"
    print(report["model_id"], "→", path)


## Metric interpretation

- `null`은 해당 cohort에서 정의 불가한 값이다. `0`으로 바꾸지 않는다.
- `test_negative_case_fp_rate`는 negative case가 실제 포함된 CT/MRI test cohort에서만 계산된다.
- ASSD/HD95는 GT 또는 prediction foreground가 비어 있는 case에서는 정의할 수 없어 해당 case를 평균에서 제외한다.
- lesion match는 26-connectivity 3-D component가 1 voxel 이상 겹치는 기준이다.
